# The drug panel — FDA-approved anticancer drugs that CTRPv2 screened

**What defines the panel.** A drug is a candidate if the FDA has approved it for a cancer indication
and CTRPv2 screened it. Nothing about our own response values enters the criterion.

That is a deliberate reversal. The two previous panels were both selected using our labels and both
were voided for it — the [learnability gate](../docs/steps/corrections-and-dead-ends.md#the-learnability-gate-measured-potency-not-rankability),
which filtered on absolute potency and so discarded every cytostatic compound, and the
[8-drug literature panel](../docs/steps/corrections-and-dead-ends.md#the-8-drug-literature-panel-and-every-number-computed-on-it),
whose candidate list was pre-ranked on our AUCs before any citation was consulted. A criterion that
touches the labels cannot be applied without seeing held-out ones, and a panel enriched for drugs that
happen to separate *our* 181 cell lines flatters every number computed on it.

**The list.** Sun, J. *et al.* A systematic analysis of FDA-approved anticancer drugs. *BMC Systems
Biology* **11**(Suppl 5), 87 (2017), doi:10.1186/s12918-017-0464-7 — Table 1, 150 drugs approved
1949–2014, with approval year, therapeutic class, target gene and delivery type. Retrieved and
verified by `scripts/sources/fetch_sun2017_drugs.py`.

**This notebook does not select.** It produces the candidate set and shows what it contains. Which of
the candidates become the panel — and whether coverage or anything else narrows them — is a separate
decision, taken after looking at this.

**It reads no pipeline artifact.** The response file is DrEval's `CTRPv2.csv` directly, so this runs
before the sweep regenerates anything and its output does not go stale when the h5ads do.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.annotation.drug_annotation import (
    annotate_compounds,
    ctrp_compounds,
    match_external_list,
)
from scripts.sources.fetch_sun2017_drugs import fetch_sun2017_drugs
from scripts.layout import PipelinePaths

OUT = ROOT / 'notebooks' / 'outputs' / 'panel'
OUT.mkdir(parents=True, exist_ok=True)

paths = PipelinePaths.build(None)
print(paths.ctrp_response_csv)

/Users/selin/Desktop/OncoTox/data/metadata/drevalpy_CTRPv2_zenodo_21807175/CTRPv2/CTRPv2.csv


## 1 · The external criterion

Table 1 is the paper's entire dataset — it publishes no supplementary file. The fetch is verified
against the counts the paper states in its Results (150 drugs, 61 cytotoxic, 89 targeted) rather than
against a checksum, which PMC does not publish for a rendered article.

**The `pubchem_cids` column is not from the paper.** Sun *et al.* name drugs by INN, and CTRPv2
screened many of them under a development code, so the names alone cannot connect the two. The CIDs
were resolved separately from PubChem and are recorded as such in the provenance file next to the CSV.

**30 of the 150 resolve to nothing, and that is the correct answer:** monoclonal antibodies, enzymes
such as pegaspargase, the cell therapy sipuleucel-T and radium-223. They have no small-molecule
structure to register and cannot appear in a small-molecule screen, so the reachable list is 120.

**22 are named in their salt form** — `Imatinib mesylate`, `Doxorubicin hydrochloride`,
`Vincristine sulfate` — because that is the form the FDA approved. A screen dissolves and names the
free base. These are distinct molecules with distinct PubChem records, so §3 resolves both sides to
the parent compound before comparing them.

In [2]:
sun = pd.read_csv(fetch_sun2017_drugs(ROOT / 'reference'), dtype={'pubchem_cids': str}).fillna(
    {'pubchem_cids': ''})

small_molecule = sun.pubchem_cids != ''
print(f'{len(sun)} FDA-approved anticancer drugs, {sun.approval_year.min()}-{sun.approval_year.max()}')
print(sun.drug_class.value_counts().to_string())
print(f'\nwith a PubChem structure  {int(small_molecule.sum()):4d}   <- reachable by a cell-line screen')
print(f'biologics, no structure   {int((~small_molecule).sum()):4d}')
sun[~small_molecule].drug.tolist()

  sun2017_fda_anticancer_drugs.csv: cached, 150 drugs {'Targeted': 89, 'Cytotoxic': 61} -- skipping fetch
150 FDA-approved anticancer drugs, 1949-2014
drug_class
Targeted     89
Cytotoxic    61

with a PubChem structure   120   <- reachable by a cell-line screen
biologics, no structure     30


['Asparaginase',
 'Pegaspargase',
 'Asparaginase erwinia chrysanthemi',
 'Radium 223 dichloride',
 'Aldesleukin',
 'Porfimer',
 'Rituximab',
 'Trastuzumab',
 'Denileukin diftitox',
 'Gemtuzumab ozogamicin',
 'Alemtuzumab',
 'Peginterferon Alfa-2b',
 'Ibritumomab tiuxetan',
 'Tositumomab and Iodine I 131 Tositumomab',
 'Bevacizumab',
 'Cetuximab',
 'Panitumumab',
 'Ofatumumab',
 'Denosumab',
 'Sipuleucel-T',
 'Brentuximab vedotin',
 'Ipilimumab',
 'Pertuzumab',
 'Ziv-aflibercept',
 'Obinutuzumab',
 'Ado-trastuzumab emtansine',
 'Pembrolizumab',
 'Ramucirumab',
 'Blinatumomab',
 'Nivolumab']

## 2 · CTRPv2's 545 compounds, joined on `master_cpd_id`

Two files describe the same 545 compounds: DrEval's `CTRPv2.csv` holds the response values, and
`data/drug/all_sources_drug_catalog.csv` — built from CTRP's own `v20.meta.per_compound.txt` in
[`drug_catalog.ipynb`](analysis/harmonization/drug_catalog.ipynb) — holds approval status, protein
target and mechanism. **They name the compounds differently**, so joining them by name loses 102 of
545, of which 15 are single-agent and FDA-approved or clinical.

`master_cpd_id` is CTRP's own compound identifier and is present in both: 545/545, nothing unmatched.
The join is asserted in `drug_annotation.annotate_compounds` and raises if it is ever inexact.

Note this is the opposite conclusion to the **cell-line** join, where names beat Cellosaurus
accessions 180 to 172 and the accession rides along as an attribute
([Step 01](../docs/steps/01-datasets-and-harmonization.md)). Each join is decided on its own
evidence; neither "always use the identifier" nor "always use the name" would have got both right.

In [3]:
compounds = annotate_compounds(ctrp_compounds(paths.ctrp_response_csv),
                               ROOT / 'data' / 'drug' / 'all_sources_drug_catalog.csv')

disagree = compounds[compounds.drug != compounds.drug_ctrp]
lost = disagree[disagree.compound_status.isin(['FDA', 'clinical']) & ~disagree.is_combination]
print(f'{len(compounds)} compounds | {int(compounds.is_combination.sum())} combinations')
print(f'names disagree between the two files: {len(disagree)}')
print(f'  of those, single-agent FDA/clinical: {len(lost)}   <- silently dropped by a name join')
lost[['drug_ctrp', 'drug', 'compound_status']].rename(
    columns={'drug_ctrp': 'CTRP name', 'drug': 'DrEval name'})

545 compounds | 49 combinations
names disagree between the two files: 102
  of those, single-agent FDA/clinical: 15   <- silently dropped by a name join


,CTRP name,DrEval name,compound_status
master_cpd_id,,,
25344,fluorouracil,5-fluorouracil,FDA
50732,mitomycin,mitomycin-c,FDA
54210,lbh-589,panobinostat,clinical
56703,sirolimus,rapamycin,FDA
340501,vx-680,tozasertib,clinical
348992,nvp-bez235,dactolisib,clinical
411808,tg-101348,fedratinib,clinical
417417,tipifarnib-p1,tipifarnib s enantiomer,clinical
417979,ym-155,sepantronium bromide,clinical


## 3 · The intersection — the candidate set

`match_external_list` tries four keys per drug and records each that succeeded, because they are not
equally strong and the panel should not hide which compound rests on which.

DrEval's spelling is the only one that finds `idelalisib`; CTRP's is the only one that finds
`fluorouracil` and `mitomycin`; the PubChem structure is the only thing connecting `Vemurafenib` to
the `plx-4032` CTRPv2 screened it as; and **the parent relation** is what connects `Imatinib mesylate`
to `imatinib`. That last key alone accounts for twelve drugs — imatinib, doxorubicin, vincristine and
topotecan among them — which the first three miss entirely on a suffix.

The expansion runs on **both** sides. CTRPv2 screened `cytarabine hydrochloride` while the list names
`Cytarabine`, so resolving only the external list would still lose it.

Every row of the list is kept, matched or not, so what the criterion *failed* to find is as visible as
what it found.

In [4]:
matched = match_external_list(compounds, sun, name_col='drug', cids_col='pubchem_cids',
                              parent_cache=ROOT / 'reference' / 'pubchem_parent_cids.csv')
candidates = matched[matched.master_cpd_id.notna()].join(
    compounds[['drug', 'compound_status', 'target', 'moa_or_pathway', 'top_test_conc_umol']],
    on='master_cpd_id', rsuffix='_ctrp')

print(f'{len(candidates)} of {len(sun)} FDA-approved anticancer drugs were screened by CTRPv2 '
      f'({len(candidates)} of {int(small_molecule.sum())} small molecules)')
print(candidates.drug_class.value_counts().to_string())
print('\nmatched by which key:')
print(candidates.matched_by.value_counts().to_string())

candidates.to_csv(OUT / 'literature_panel_candidates.csv', index=False)
candidates[['drug', 'approval_year', 'drug_class', 'target_gene', 'therapeutic_class',
            'compound_status', 'matched_by']].sort_values(['drug_class', 'approval_year'])

57 of 150 FDA-approved anticancer drugs were screened by CTRPv2 (57 of 120 small molecules)
drug_class
Targeted     32
Cytotoxic    25

matched by which key:
matched_by
name_dreval+name_ctrp+pubchem_cid    39
pubchem_parent                       13
name_ctrp+pubchem_cid                 2
pubchem_cid                           2
name_dreval+pubchem_cid               1


,drug,approval_year,drug_class,target_gene,therapeutic_class,compound_status,matched_by
2,Methotrexate,1953,Cytotoxic,DHFR,Leukemia; Breast cancer; Head and neck cancer;...,FDA,name_dreval+name_ctrp+pubchem_cid
5,Chlorambucil,1957,Cytotoxic,DNA synthesis,Leukemia; Lymphoma,FDA,name_dreval+name_ctrp+pubchem_cid
6,Cyclophosphamide,1959,Cytotoxic,DNA synthesis,Lymphoma; Multiple myeloma; Leukemia; Brain ca...,FDA,name_dreval+name_ctrp+pubchem_cid
7,Vincristine sulfate,1963,Cytotoxic,TUBA4A; TUBB,Leukemia,FDA,pubchem_parent
11,Procarbazine hydrochloride,1969,Cytotoxic,DNA synthesis,Lymphoma,FDA,pubchem_parent
13,Fluorouracil,1970,Cytotoxic,DNA synthesis,Breast cancer; Colorectal cancer; Stomach canc...,FDA,name_ctrp+pubchem_cid
16,Doxorubicin hydrochloride,1974,Cytotoxic,TOP2A; DNA synthesis,Leukemia; Breast cancer; Stomach cancer; Lymph...,FDA,pubchem_parent
17,Dacarbazine,1975,Cytotoxic,DNA synthesis,Melanoma; Lymphoma,FDA,name_dreval+name_ctrp+pubchem_cid
20,Cisplatin,1978,Cytotoxic,DNA synthesis,Testicular cancer; Ovarian cancer; Bladder cancer,FDA,pubchem_cid
23,Etoposide,1983,Cytotoxic,TOP2A; TOP2B,Testicular cancer; Lung cancer,FDA,name_dreval+name_ctrp+pubchem_cid


## 4 · Coverage — context for narrowing, not a criterion

A candidate is only usable to the extent CTRPv2 screened it against cell lines **we have expression
for**. Coverage is therefore counted over the **181-line SCP542 ∩ CTRPv2 overlap**, not over CTRPv2's
886 lines: a compound screened against 800 lines that misses ours trains nothing.

The overlap comes from `ctrp_to_h5ad.overlap_cell_lines`, the same function the preprocessing step
uses, so this cannot disagree with what the pipeline builds. It reads the raw SCP542 h5ad for its cell
roster only — that file predates the gene-symbol repair, which changes genes and leaves `obs`
untouched, so the roster is current even though the matrix is not.

**Both response measures are counted.** `auc_cc` is the target; `ln_ic50_cc` is absent for ~40 % of
curves by construction, because DrEval discard an IC50 falling more than an order of magnitude outside
the measured dose range. A compound can therefore have full `auc_cc` coverage and little `ln_ic50_cc`
coverage, which matters for any comparison run on both.

Nothing is filtered here.

In [5]:
import anndata as ad

from scripts.preprocessing.ctrp_to_h5ad import (
    _deduplicate_measurements,
    _load_drevalpy_long,
    overlap_cell_lines,
)

roster = ad.read_h5ad(paths.raw_h5ad, backed='r').obs['Cell_line']   # backed: .X stays on disk

coverage = {}
for score in ('auc_cc', 'ln_ic50_cc'):
    long = _deduplicate_measurements(_load_drevalpy_long(paths.ctrp_response_csv, score))
    _, overlap = overlap_cell_lines(roster, long)
    coverage[score] = (long[long.ccl_name_norm.isin(overlap)]
                       .groupby('cpd_name_norm').ccl_name_norm.nunique())
    print(f'{score}: {len(overlap)} overlapping cell lines')

N_LINES = len(overlap)
panel = candidates.join(compounds[['drug']], on='master_cpd_id', rsuffix='_key')
panel['n_auc_cc'] = panel.drug_key.map(coverage['auc_cc']).fillna(0).astype(int)
panel['n_ln_ic50_cc'] = panel.drug_key.map(coverage['ln_ic50_cc']).fillna(0).astype(int)
panel['coverage'] = panel.n_auc_cc / N_LINES

print(f'\ncoverage of the {N_LINES} overlapping lines, across the {len(panel)} candidates:')
print(panel.coverage.describe()[['min', '25%', '50%', '75%', 'max']].round(3).to_string())
for threshold in (1.00, 0.95, 0.90, 0.50):
    print(f'  >= {threshold:.0%}: {int((panel.coverage >= threshold).sum()):2d} candidates')

panel.to_csv(OUT / 'literature_panel_candidates.csv', index=False)
panel[['drug', 'drug_key', 'approval_year', 'drug_class', 'target_gene',
       'n_auc_cc', 'n_ln_ic50_cc', 'coverage']].sort_values('coverage', ascending=False)

  395,024 measurements | 886 cell lines | 545 drugs
  8,187 of 395,024 rows are exact duplicates of another row (2.1 %) and are dropped.
auc_cc: 181 overlapping cell lines


  159,050 of 395,024 rows have no LN_IC50_curvecurator (40.3 %) and are dropped.
  235,974 measurements | 886 cell lines | 545 drugs
  4,346 of 235,974 rows are exact duplicates of another row (1.8 %) and are dropped.
ln_ic50_cc: 181 overlapping cell lines

coverage of the 181 overlapping lines, across the 57 candidates:
min    0.287
25%    0.950
50%    0.961
75%    0.972
max    0.994
  >= 100%:  0 candidates
  >= 95%: 43 candidates
  >= 90%: 45 candidates
  >= 50%: 51 candidates


,drug,drug_key,approval_year,drug_class,target_gene,n_auc_cc,n_ln_ic50_cc,coverage
102,Decitabine,decitabine,2006,Targeted,DNMT1,180,5,0.994475
7,Vincristine sulfate,vincristine,1963,Cytotoxic,TUBA4A; TUBB,179,104,0.988950
93,Gefitinib,gefitinib,2003,Targeted,EGFR,179,153,0.988950
112,Pazopanib hydrochloride,pazopanib,2009,Targeted,FGF1; FGFR3; FLT1; FLT4; ITK; KDR; KIT; PDGFRA...,179,89,0.988950
101,Dasatinib,dasatinib,2006,Targeted,BCR-ABL,178,172,0.983425
37,Gemcitabine,gemcitabine,1996,Cytotoxic,DNA synthesis; RRM1; TYMS,178,96,0.983425
125,Axitinib,axitinib,2012,Targeted,FLT1; FLT4; KDR,178,100,0.983425
51,Clofarabine,clofarabine,2004,Cytotoxic,DNA synthesis,178,69,0.983425
25,Carboplatin,carboplatin,1989,Cytotoxic,DNA synthesis,177,22,0.977901
121,Ruxolitinib phosphate,ruxolitinib,2011,Targeted,JAK1; JAK2,177,158,0.977901


## 5 · The panel

Two conditions, and they are of different kinds. **Coverage is derived** — the ≥ 90 % cut is where §4's
distribution breaks, and the break is sharp: 45 candidates at or above it, the lowest `afatinib`
at 91.2 %, against 69.1 % for the next compound down and 28.7 % at the tail. **The literature judgement cannot
be derived** and is carried below as a table of one claim per drug with its reference, so the evidence
travels with the list instead of living in a conversation. The code asserts every entry lies inside the
candidate set of §3, so a drug cannot enter the panel without passing the FDA criterion first.

**Nine come from the four papers this project is built on** — verified against the PDFs, not from a
name scan. Two candidates that a string search had flagged were dropped on inspection: *doxorubicin*'s
mentions in Kinker are the names of published **senescence gene programs** ("lung cancer doxorubicin"),
and *carboplatin*'s only mention is a title in scDEAL's bibliography.

**Two were added to fill the class spread** — an anthracycline and a platinum, the two most recognisable
cytotoxic classes and both absent from the nine. They enter on external determinants rather than on our
papers, which is recorded in the `setting` column rather than blurred.

**The strength column is not decoration.** It says what each citation actually establishes. Four entries
are mechanistic worked examples rather than expression–sensitivity claims; one (`erlotinib`) is a
benchmark drug with no determinant claim attached at all; and one (`cisplatin`) is **contested**. None of
that disqualifies a drug under a criterion whose first condition is FDA approval — but a panel where
every drug had a known strong expression marker would be the stacked deck this rebuild exists to avoid.

**Cisplatin's entry states the contest rather than picking a side**, because neither side wins on the
usual tie-breaks: the negative study is the more recent one *and* the far less cited one (28 vs 155),
and it is a different lineage and assay, so it does not supersede the positive. What actually explains
the disagreement is Friboulet's finding that only one of ERCC1's four splice isoforms repairs DNA. That
is not a caveat we can engineer away — our features are **gene-level CPM**, so the functional isoform is
not representable in them. If the model ranks cell lines poorly on cisplatin, this is the first thing to
look at, and it was known before the run rather than after it.

**One platinum, deliberately.** CTRPv2 screens cisplatin, carboplatin and oxaliplatin, all with ~176 of
181 lines. Cisplatin and carboplatin share the *cis*-diammine carrier and form the same 1,2-intrastrand
GpG adduct, so they are largely cross-resistant and driven by the same repair biology; a second one
would spend a panel slot on the same resistance mechanism. Oxaliplatin's DACH–Pt adduct evades mismatch
repair and is genuinely complementary, but its expression determinant is the least established of the
three. The class is therefore represented once, by cisplatin.

**`cisplatin` is `platin` in the data.** CTRPv2 names it that, and the response file's `pubchem_id` for
it is *elemental platinum*; `drug_annotation.CTRP_PUBCHEM_OVERRIDES` corrects it to CID 5702198 on the
evidence of CTRP's own SMILES `N[Pt](N)(Cl)Cl` and vendor entry Selleck S1166. The panel is keyed by the
name the data uses, so downstream code needs no translation table.

In [6]:
MIN_COVERAGE = 0.90        # the break in the coverage distribution, see section 4

# drug key (as CTRPv2/DrEval name it) -> (reference, what the reference actually claims, setting, strength)
PANEL_EVIDENCE = {
 'gemcitabine': (
    'Rees et al., Nat Chem Biol 12:109-116 (2016)',
    'basal expression correlates with sensitivity across CTRPv2 lines (r = -0.39, p = 3.6e-23), on the '
    'SLFN11 axis it shares with topoisomerase-I inhibitors',
    'expression-sensitivity, this screen', 'strong'),
 'imatinib': (
    'Rees et al., Nat Chem Biol 12:109-116 (2016); Skinner, Palkar & Hong, Cancers 15:4236 (2023)',
    'high ABL1 and PDGFRA expression correlate with imatinib sensitivity; ABL1 uniquely associated in '
    'haematologic lines. Separately a named P-glycoprotein substrate (Skinner), so ABCB1 expression is '
    'a second, independent expression-level route to resistance',
    'expression-sensitivity, this screen', 'strong'),
 'sorafenib': (
    'Kinker et al., Nat Genet 52:1208-1218 (2020)',
    'hit in the EpiSen-high differential-killing screen, annotated SLC7A11',
    'expression-linked, our own atlas', 'strong'),
 'etoposide': (
    'Omori et al., Thorac Cancer 13:2142-2151 (2022); Skinner, Palkar & Hong, Cancers 15:4236 (2023); '
    'Kinker et al., Nat Genet 52:1208-1218 (2020)',
    'ABCB1 overexpressed in etoposide-resistant SCLC lines and silencing it restores sensitivity, and '
    'etoposide is a canonical P-glycoprotein substrate; topoisomerase expression itself did NOT '
    'correlate with sensitivity, so TOP2A is deliberately not claimed. Also used by Kinker to induce '
    'senescence in primary lung bronchial cells -- a reagent use, not a response claim, recorded '
    'because it is a mention in our own atlas paper and should not be mistaken for the determinant',
    'expression determinant, 12 external cell lines, causal by silencing', 'strong'),
 'crizotinib': (
    'Seashore-Ludlow et al., Cancer Discov 5:1210-1223 (2015)',
    'ALK-inhibitor worked example in neuroblastoma; IC50 shifts under ALK/IGF1R co-treatment',
    'mechanistic worked example, this screen', 'medium'),
 'paclitaxel': (
    'Seashore-Ludlow et al., Cancer Discov 5:1210-1223 (2015); Skinner, Palkar & Hong, Cancers '
    '15:4236 (2023)',
    'member of the mitosis/microtubule compound cluster, confirmed in a tubulin-polymerization assay. '
    'Separately a named P-glycoprotein substrate (Skinner), which is an expression-level determinant '
    'the Seashore-Ludlow mention does not supply',
    'mechanistic worked example, this screen', 'medium'),
 'dasatinib': (
    'Seashore-Ludlow et al., Cancer Discov 5:1210-1223 (2015)',
    'exemplary compound of the SRC target cluster, chosen for the combination panel',
    'mechanistic worked example, this screen', 'medium'),
 'afatinib': (
    'Seashore-Ludlow et al., Cancer Discov 5:1210-1223 (2015)',
    'exemplary irreversible EGFR inhibitor, same combination panel',
    'mechanistic worked example, this screen', 'medium'),
 'erlotinib': (
    'Chen et al., Nat Commun 13:6494 (2022)',
    'one of the five drugs the scDEAL single-cell response benchmark is built on -- relevance to the '
    'task, NOT a determinant claim',
    'benchmark drug, no determinant claimed', 'medium'),
 'doxorubicin': (
    'Skinner, Palkar & Hong, Cancers 15:4236 (2023)',
    'canonical P-glycoprotein substrate; ABCB1 overexpression prevents intracellular accumulation and '
    'mediates resistance',
    'expression determinant, external -- review, not a primary measurement', 'medium'),
 'platin': (
    'Chen et al., Nat Commun 13:6494 (2022); Kinker et al., Nat Genet 52:1208-1218 (2020). '
    'Determinant: Britten et al., Int J Cancer 89:453-457 (2000); Shimizu et al., Respirology '
    '13:510-517 (2008); Friboulet et al., N Engl J Med 368:1101-1110 (2013)',
    'Named in two of this project\'s own papers -- one of the five drugs the scDEAL benchmark is built '
    'on, and a perturbation in Kinker Fig. 6, tested for association with expression heterogeneity '
    'alongside etoposide. Its response DETERMINANT, however, is CONTESTED and unresolved at the level '
    'this project measures. Britten (155 citations): ERCC1 mRNA '
    'correlates with cisplatin resistance in cervical carcinoma lines (p <= 0.011) while ERCC1 protein '
    'does not. Shimizu (28 citations): no correlation between ERCC1 mRNA and cisplatin or carboplatin '
    'sensitivity across 20 lung cancer lines -- more recent, far less cited, and a different lineage '
    'and assay, so it does not supersede Britten. Friboulet, on 494 patients across two phase 3 '
    'trials, explains why the field never converged: ERCC1 has four splice isoforms, only ERCC1-202 '
    'repairs DNA, and available antibodies cannot distinguish them. Gene-level expression -- ours '
    'included -- cannot represent that distinction at all',
    'expression determinant, external, contested and isoform-confounded', 'contested'),
}

outside = sorted(set(PANEL_EVIDENCE) - set(panel.drug_key))
assert not outside, f'panel drugs outside the derived candidate set: {outside}'

selected = panel.drug_key.isin(PANEL_EVIDENCE) & (panel.coverage >= MIN_COVERAGE)
assert int(selected.sum()) == len(PANEL_EVIDENCE), (
    f'{len(PANEL_EVIDENCE) - int(selected.sum())} panel drugs fail the coverage cut')

PANEL = panel[selected].copy()
PANEL['reference'] = PANEL.drug_key.map(lambda d: PANEL_EVIDENCE[d][0])
PANEL['claim'] = PANEL.drug_key.map(lambda d: PANEL_EVIDENCE[d][1])
PANEL['setting'] = PANEL.drug_key.map(lambda d: PANEL_EVIDENCE[d][2])
PANEL['strength'] = PANEL.drug_key.map(lambda d: PANEL_EVIDENCE[d][3])
PANEL = PANEL.sort_values(['drug_class', 'approval_year'])
PANEL.to_csv(OUT / 'panel.csv', index=False)

print(f'panel: {len(PANEL)} drugs from {len(panel)} candidates from {len(sun)} FDA-approved drugs')
print(PANEL.drug_class.value_counts().to_string())
print(PANEL.strength.value_counts().to_string())
print(f'\ncoverage {PANEL.coverage.min():.1%} - {PANEL.coverage.max():.1%} of {N_LINES} lines '
      f'| ln_ic50_cc {PANEL.n_ln_ic50_cc.min()} - {PANEL.n_ln_ic50_cc.max()}')
print('\ntarget_drugs for ctrp_to_h5ad / --drugs:')
print(' '.join(PANEL.drug_key))
PANEL[['drug', 'drug_key', 'drug_class', 'target_gene', 'coverage', 'n_ln_ic50_cc', 'setting', 'strength']]

panel: 11 drugs from 57 candidates from 150 FDA-approved drugs
drug_class
Targeted     6
Cytotoxic    5
strength
medium       6
strong       4
contested    1

coverage 91.2% - 98.3% of 181 lines | ln_ic50_cc 14 - 174

target_drugs for ctrp_to_h5ad / --drugs:
doxorubicin platin etoposide paclitaxel gemcitabine imatinib erlotinib sorafenib dasatinib crizotinib afatinib


,drug,drug_key,drug_class,target_gene,coverage,n_ln_ic50_cc,setting,strength
16,Doxorubicin hydrochloride,doxorubicin,Cytotoxic,TOP2A; DNA synthesis,0.977901,172,"expression determinant, external -- review, no...",medium
20,Cisplatin,platin,Cytotoxic,DNA synthesis,0.966851,14,"expression determinant, external, contested an...",contested
23,Etoposide,etoposide,Cytotoxic,TOP2A; TOP2B,0.955801,124,"expression determinant, 12 external cell lines...",strong
29,Paclitaxel,paclitaxel,Cytotoxic,TUBA4A; TUBB1,0.955801,80,"mechanistic worked example, this screen",medium
37,Gemcitabine,gemcitabine,Cytotoxic,DNA synthesis; RRM1; TYMS,0.983425,96,"expression-sensitivity, this screen",strong
86,Imatinib mesylate,imatinib,Targeted,BCR-ABL,0.972376,100,"expression-sensitivity, this screen",strong
97,Erlotinib hydrochloride,erlotinib,Targeted,EGFR,0.972376,81,"benchmark drug, no determinant claimed",medium
100,Sorafenib tosylate,sorafenib,Targeted,BRAF; FGFR1; FLT1; FLT3; FLT4; KDR; KIT; PDGFR...,0.955801,118,"expression-linked, our own atlas",strong
101,Dasatinib,dasatinib,Targeted,BCR-ABL,0.983425,172,"mechanistic worked example, this screen",medium
119,Crizotinib,crizotinib,Targeted,ALK; MET,0.961326,174,"mechanistic worked example, this screen",medium
